# 07_01_グループ内の相対アドバンテージ

コード：`ch07/blocks/07_01_グループ内の相対アドバンテージ.ipynb`

本文のコードをコードセルで直接実行します。

上からセルを実行してください。ローカルではコード一式と同じPython環境をJupyterのカーネルに選びます。Colabでコード一式が必要な場合は、準備セルで配布ZIPをアップロードします。実行確認の範囲は `VALIDATION_COLAB_20261110.md` を参照してください。


In [ ]:
import subprocess
import sys

# 既に導入済みなら不要な更新をせず、満たしていない依存だけを導入する。
subprocess.run([sys.executable, "-m", "pip", "install", *['numpy>=1.26,<3', 'torch>=2.6,<3', 'torchvision>=0.21,<1', 'mcp==2.3.0', 'PyYAML==6.0.3', 'opentelemetry-sdk==1.45.2', 'opentelemetry-exporter-otlp-proto-http==1.45.2']], check=True)


In [ ]:
# 7 章：GRPO のアドバンテージ。1 問ごとに G 本生成し、報酬をグループの中で標準化する
import numpy as np

def grpo_advantages(rewards, eps=1e-6):
    r = np.asarray(rewards, dtype=float)
    return (r - r.mean()) / (r.std() + eps)       # ★ 価値関数（critic）を使わず、同じ問題の仲間と比べる

groups = {"問 1（4 本中 1 本正解）": [1, 0, 0, 0],
          "問 2（4 本中 3 本正解）": [1, 1, 1, 0],
          "問 3（全部正解）":        [1, 1, 1, 1],
          "問 4（全部不正解）":      [0, 0, 0, 0]}
for name, r in groups.items():
    print(f"{name}  報酬 {r}  アドバンテージ {np.round(grpo_advantages(r), 2).tolist()}")

# 方策の更新量：PPO と同じクリップつきの比。比 = 新しい方策の確率 / 生成したときの方策の確率
def clipped_objective(ratio, adv, clip=0.2):
    return np.minimum(ratio * adv, np.clip(ratio, 1 - clip, 1 + clip) * adv)

adv = grpo_advantages([1, 0, 0, 0])
for ratio in (0.7, 1.0, 1.3):
    obj = clipped_objective(ratio, adv)
    print(f"比 {ratio}：正解の解答の目的値 {obj[0]:+.2f}  不正解の解答の目的値 {obj[1]:+.2f}")
